<a href="https://colab.research.google.com/github/Cd881/CSCI164/blob/main/Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Orignal Code**: [github.com/everestso/Summer24/blob/main/AI24Ch3a.ipynb](https://github.com/everestso/Summer24/blob/main/AI24Ch3a.ipynb)

In [1]:
import random
import heapq

# Tile Sliding Domain

## Functions

In [2]:
def RandomWalk(state, steps):
  actionSequence = []
  actionLast = None
  for i in range(steps):
    action = None
    while action==None:
      action = random.choice(Actions(state))
      action = action if (LegalMove(state, action)
          and action!= Opposite[actionLast]) else None
    actionLast = action
    state = Result(state, action)
    actionSequence.append(action)
  return state, actionSequence

In [3]:
def ApplyMoves(actions, state):
  for action in actions:
    state = Result(state, action)
  return state

In [4]:
def ReverseMoves(actions):
  ret = [Opposite[a] for a in actions]
  ret.reverse()
  return ret

In [5]:
class Problem(object): pass

In [6]:
class Node(object):
  def __init__(self, state, parent=None, action=None, path_cost=0 ):
    self.State=state
    self.Parent=parent
    self.Action=action
    self.PathCost = path_cost

  def __str__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __repr__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __lt__(self, other):
    return self.PathCost < other.PathCost;

In [7]:
def Expand(problem, node):
  ret = []
  s = node.State
  for action in problem.Actions(s):
    sPrime = problem.Result(s, action)
    cost =node.PathCost + problem.ActionCost(s,action,sPrime)
    ret.append(Node(sPrime, node, action, cost))
  return ret

In [8]:
def BreadthFirstSearch(problem):
  node = Node(tuple(problem.INITIAL))
  if problem.IsGoal(node.State):
    return node, 0
  Frontier = []
  Frontier.append(node)
  reached = set()
  reached.add(tuple(problem.INITIAL))
  nodesExpanded = 0
  while (Frontier):
    ### print([str(n) for n in Frontier])
    node = Frontier.pop(0)
    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      ### print (s, "IsGoal=", problem.IsGoal(s))
      if problem.IsGoal(s):
        return child, nodesExpanded
      if s not in reached:
        reached.add(s)
        Frontier.append(child)
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

In [9]:
def BestFirstSearch(problem, f):
  node = Node(tuple(problem.INITIAL))
  Frontier = []
  heapq.heappush(Frontier,(f(node), node))
  reached = {}
  reached[tuple(problem.INITIAL)]=node
  nodesExpanded = 0
  while (Frontier):
    ##print([(x, str(n)) for (x,n) in Frontier])
    fValue, node = heapq.heappop(Frontier)
    ##print (node.State, "IsGoal=", problem.IsGoal(tuple(node.State)))
    if problem.IsGoal(tuple(node.State)):
      return node, nodesExpanded    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      if s not in reached or child.PathCost < reached[s].PathCost:
        reached[s] = child
        heapq.heappush(Frontier, (f(child), child))
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

In [67]:
def Solution(node):
  if node==None:
    return []
  if node.Parent==None:
    return []
  return Solution(node.Parent) + [node.Action]

In [11]:
def GeneratePuzzle(findNum, dist, goalState):
  problemList = []
  for i in range(findNum):
    state1, sol = RandomWalk(goalState, dist)
    problemList.append(state1)
  return problemList

## 3x3 Puzzle

In [12]:
StateDimension=3
InitialState = [1,2,3,4,5,6,0,7,8]
GoalState=[1,2,3,4,5,6,7,8,0]
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite=dict([('u','d'),('d','u'),('l','r'),('r','l'), (None, None)])

In [13]:
def Result(state, action):
  i = state.index(0)
  newState = list(state)
  row,col=i//StateDimension, i % StateDimension
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return newState
  if action=='u':
    l,r = row*StateDimension+col, (row-1)*StateDimension+col
  elif action=='d':
    l,r = row*StateDimension+col, (row+1)*StateDimension+col
  elif action=='l':
    l,r = row*StateDimension+col, row*StateDimension+col-1
  elif action=='r' :
    l,r = row*StateDimension+col, row*StateDimension+col+1
  newState[l], newState[r] = newState[r], newState[l]
  return newState

def PrintState(s):
  for i in range(0,len(s),StateDimension):
    print(s[i:i+StateDimension])

def LegalMove(state, action):
  i = state.index(0)
  row,col=i//StateDimension, i % StateDimension
  newState = state.copy()
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return False
  return True

In [14]:
def SingleTileManhattanDistance(tile, left, right):
  leftIndex = left.index(tile)
  rightIndex = right.index(tile)
  return (abs(leftIndex//StateDimension-rightIndex//StateDimension) +
          abs(leftIndex%StateDimension-rightIndex%StateDimension))

def ManhattanDistance(left, right):
  distances = [SingleTileManhattanDistance(tile, left, right)
     for tile in range(1, StateDimension**2)]
  ### print ("Distances= ", distances)
  return sum(distances)


In [15]:
def OutOfPlace(left, right):
  distances = [left[i]!=right[i] and right[i] != 0
     for i in range(StateDimension**2)]
  return sum(distances)

In [16]:
PrintState(InitialState)

[1, 2, 3]
[4, 5, 6]
[0, 7, 8]


In [17]:
PrintState(GoalState)

[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


In [18]:
print("ManhattanDistance=  ", ManhattanDistance(InitialState, GoalState))
print("OutOfPlace= ", OutOfPlace(InitialState, GoalState))

ManhattanDistance=   2
OutOfPlace=  2


In [19]:
PrintState(InitialState)
print()
state1 = Result(InitialState, 'u')
PrintState(state1)
print()
state1 = Result(state1, 'r')
PrintState(state1)

[1, 2, 3]
[4, 5, 6]
[0, 7, 8]

[1, 2, 3]
[0, 5, 6]
[4, 7, 8]

[1, 2, 3]
[5, 0, 6]
[4, 7, 8]


## Tests

### Generation of Puzzles

In [20]:
randomWalk_5step = GeneratePuzzle(3, 5, GoalState)
print(randomWalk_5step)

[[4, 1, 2, 0, 5, 3, 7, 8, 6], [1, 2, 3, 0, 8, 5, 4, 7, 6], [1, 2, 3, 0, 8, 5, 4, 7, 6]]


In [21]:
randomWalk_10step = GeneratePuzzle(3, 10, GoalState)
print(randomWalk_10step)

[[2, 4, 3, 7, 0, 6, 5, 1, 8], [0, 5, 3, 2, 7, 6, 1, 4, 8], [5, 3, 0, 2, 1, 6, 4, 7, 8]]


In [22]:
randomWalk_20step = GeneratePuzzle(3, 20, GoalState)
print(randomWalk_20step)

[[7, 2, 3, 1, 8, 6, 0, 5, 4], [4, 1, 6, 5, 2, 3, 0, 7, 8], [8, 4, 3, 5, 2, 1, 7, 6, 0]]


In [23]:
randomWalk_40step = GeneratePuzzle(3, 40, GoalState)
print(randomWalk_40step)

[[7, 8, 4, 5, 0, 2, 1, 6, 3], [6, 8, 7, 5, 2, 4, 0, 3, 1], [4, 3, 8, 1, 2, 5, 6, 7, 0]]


In [24]:
randomWalk_80step = GeneratePuzzle(3, 80, GoalState)
print(randomWalk_80step)

[[6, 3, 1, 2, 0, 8, 7, 4, 5], [8, 2, 0, 4, 3, 1, 5, 7, 6], [5, 3, 8, 7, 2, 1, 4, 6, 0]]


In [26]:
test_problems = randomWalk_5step + randomWalk_10step + randomWalk_20step + randomWalk_40step + randomWalk_80step

### Breadth-First Search

In [33]:
TileSliding = Problem()
TileSliding.INITIAL = InitialState
TileSliding.IsGoal = lambda s: s==(1,2,3,4,5,6,7,8,0)
TileSliding.Actions = Actions
TileSliding.Result=Result
TileSliding.ActionCost = lambda s, a, sPrime: 1

Solutions = []
for s in test_problems:
  TileSliding.INITIAL = s
  ret, cost = BreadthFirstSearch(TileSliding)
  sol = Solution(ret)
  print (sol, ":", TileSliding.INITIAL)
  #print ("-----------------------")
  #print (TileSliding.INITIAL,':' )
  #print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost, '\n\n')
  #print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
#print ("-------")
#print (Solutions)

['u', 'r', 'r', 'd', 'd'] : [4, 1, 2, 0, 5, 3, 7, 8, 6]
Length of solution:  5
Nodes Expanded= 22 


['d', 'r', 'u', 'r', 'd'] : [1, 2, 3, 0, 8, 5, 4, 7, 6]
Length of solution:  5
Nodes Expanded= 25 


['d', 'r', 'u', 'r', 'd'] : [1, 2, 3, 0, 8, 5, 4, 7, 6]
Length of solution:  5
Nodes Expanded= 25 


['d', 'l', 'u', 'r', 'u', 'l', 'd', 'r', 'd', 'r'] : [2, 4, 3, 7, 0, 6, 5, 1, 8]
Length of solution:  10
Nodes Expanded= 402 


['d', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'r'] : [0, 5, 3, 2, 7, 6, 1, 4, 8]
Length of solution:  10
Nodes Expanded= 268 


['l', 'd', 'l', 'u', 'r', 'd', 'l', 'd', 'r', 'r'] : [5, 3, 0, 2, 1, 6, 4, 7, 8]
Length of solution:  10
Nodes Expanded= 365 


['u', 'u', 'r', 'd', 'd', 'r', 'u', 'l', 'l', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'u', 'r', 'd'] : [7, 2, 3, 1, 8, 6, 0, 5, 4]
Length of solution:  20
Nodes Expanded= 27507 


['r', 'u', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'd'] : [4, 1, 6, 5, 2, 3, 0, 7, 8]
Length of so

### A* w/ Out of place

In [34]:
AStarFb = lambda n: n.PathCost + OutOfPlace(n.State, GoalState)

Solutions = []
for s in test_problems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarFb)
  sol = Solution(ret)
  print (sol, ":", TileSliding.INITIAL)
  #print ("-----------------------")
  #print (TileSliding.INITIAL,':' )
  #print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost, '\n\n')
  #print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
#print ("-------")
#print (Solutions)

['u', 'r', 'r', 'd', 'd'] : [4, 1, 2, 0, 5, 3, 7, 8, 6]
Length of solution:  5
Nodes Expanded= 5 


['d', 'r', 'u', 'r', 'd'] : [1, 2, 3, 0, 8, 5, 4, 7, 6]
Length of solution:  5
Nodes Expanded= 5 


['d', 'r', 'u', 'r', 'd'] : [1, 2, 3, 0, 8, 5, 4, 7, 6]
Length of solution:  5
Nodes Expanded= 5 


['d', 'l', 'u', 'r', 'u', 'l', 'd', 'r', 'd', 'r'] : [2, 4, 3, 7, 0, 6, 5, 1, 8]
Length of solution:  10
Nodes Expanded= 43 


['d', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'r'] : [0, 5, 3, 2, 7, 6, 1, 4, 8]
Length of solution:  10
Nodes Expanded= 35 


['l', 'l', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'r'] : [5, 3, 0, 2, 1, 6, 4, 7, 8]
Length of solution:  10
Nodes Expanded= 32 


['r', 'r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'u', 'u', 'r', 'd', 'd', 'l', 'u', 'l', 'd', 'r', 'r'] : [7, 2, 3, 1, 8, 6, 0, 5, 4]
Length of solution:  20
Nodes Expanded= 3406 


['r', 'u', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'd'] : [4, 1, 6, 5, 2, 3, 0, 7, 8]
Length of solution:

### A* w/ Manhattan

In [35]:
AStarF = lambda n: n.PathCost + ManhattanDistance(n.State, GoalState)

Solutions = []
for s in test_problems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarF)
  sol = Solution(ret)
  print (sol, ":", TileSliding.INITIAL)
  #print ("-----------------------")
  #print (TileSliding.INITIAL,':' )
  #print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost, '\n\n')
  #print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
#print ("-------")
#print (Solutions)

['u', 'r', 'r', 'd', 'd'] : [4, 1, 2, 0, 5, 3, 7, 8, 6]
Length of solution:  5
Nodes Expanded= 5 


['d', 'r', 'u', 'r', 'd'] : [1, 2, 3, 0, 8, 5, 4, 7, 6]
Length of solution:  5
Nodes Expanded= 5 


['d', 'r', 'u', 'r', 'd'] : [1, 2, 3, 0, 8, 5, 4, 7, 6]
Length of solution:  5
Nodes Expanded= 5 


['d', 'l', 'u', 'r', 'u', 'l', 'd', 'r', 'd', 'r'] : [2, 4, 3, 7, 0, 6, 5, 1, 8]
Length of solution:  10
Nodes Expanded= 15 


['d', 'd', 'r', 'u', 'u', 'l', 'd', 'd', 'r', 'r'] : [0, 5, 3, 2, 7, 6, 1, 4, 8]
Length of solution:  10
Nodes Expanded= 16 


['l', 'd', 'l', 'u', 'r', 'd', 'l', 'd', 'r', 'r'] : [5, 3, 0, 2, 1, 6, 4, 7, 8]
Length of solution:  10
Nodes Expanded= 21 


['r', 'u', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'l', 'u', 'r', 'r', 'u', 'l', 'l', 'd', 'r', 'd', 'r'] : [7, 2, 3, 1, 8, 6, 0, 5, 4]
Length of solution:  20
Nodes Expanded= 651 


['r', 'u', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'r', 'd'] : [4, 1, 6, 5, 2, 3, 0, 7, 8]
Length of solution: 

## 4x4 Puzzle

In [55]:
StateDimension=4
InitialState = [1,2,3,4,5,6,0,7,8,9,10,11,12,13,14,15]
GoalState=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite=dict([('u','d'),('d','u'),('l','r'),('r','l'), (None, None)])

In [56]:
PrintState(InitialState)

[1, 2, 3, 4]
[5, 6, 0, 7]
[8, 9, 10, 11]
[12, 13, 14, 15]


In [57]:
PrintState(GoalState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [58]:
print("ManhattanDistance=  ", ManhattanDistance(InitialState, GoalState))
print("OutOfPlace= ", OutOfPlace(InitialState, GoalState))

ManhattanDistance=   15
OutOfPlace=  9


In [59]:
PrintState(InitialState)
print()
state1 = Result(InitialState, 'u')
PrintState(state1)
print()
state1 = Result(state1, 'r')
PrintState(state1)

[1, 2, 3, 4]
[5, 6, 0, 7]
[8, 9, 10, 11]
[12, 13, 14, 15]

[1, 2, 0, 4]
[5, 6, 3, 7]
[8, 9, 10, 11]
[12, 13, 14, 15]

[1, 2, 4, 0]
[5, 6, 3, 7]
[8, 9, 10, 11]
[12, 13, 14, 15]


## Tests

### Puzzle generation

In [60]:
randomWalk_5step = GeneratePuzzle(3, 5, GoalState)
print(randomWalk_5step)

[[1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12], [1, 2, 3, 4, 0, 6, 7, 8, 5, 9, 10, 12, 13, 14, 11, 15], [1, 2, 3, 4, 0, 6, 7, 8, 5, 10, 11, 12, 9, 13, 14, 15]]


In [61]:
randomWalk_10step = GeneratePuzzle(3, 10, GoalState)
print(randomWalk_10step)

[[5, 1, 2, 3, 9, 6, 7, 4, 10, 11, 0, 8, 13, 14, 15, 12], [5, 1, 0, 4, 2, 6, 3, 8, 9, 10, 7, 12, 13, 14, 11, 15], [1, 2, 3, 4, 9, 5, 6, 7, 13, 11, 0, 8, 14, 10, 15, 12]]


In [62]:
randomWalk_20step = GeneratePuzzle(3, 20, GoalState)
print(randomWalk_20step)

[[0, 3, 7, 4, 2, 6, 5, 8, 1, 13, 10, 11, 14, 9, 15, 12], [1, 2, 4, 8, 5, 6, 7, 3, 9, 10, 0, 11, 13, 14, 15, 12], [1, 7, 0, 3, 10, 2, 6, 4, 5, 14, 11, 8, 9, 13, 15, 12]]


In [63]:
randomWalk_40step = GeneratePuzzle(3, 40, GoalState)
print(randomWalk_40step)

[[1, 2, 6, 3, 5, 15, 4, 11, 0, 13, 10, 8, 7, 9, 14, 12], [2, 6, 3, 4, 13, 1, 7, 0, 9, 8, 15, 5, 14, 10, 12, 11], [1, 3, 8, 7, 2, 5, 6, 4, 0, 9, 12, 15, 13, 10, 14, 11]]


In [64]:
randomWalk_80step = GeneratePuzzle(3, 80, GoalState)
print(randomWalk_80step)

[[2, 6, 3, 4, 1, 5, 8, 9, 14, 12, 0, 10, 7, 11, 15, 13], [14, 3, 2, 4, 9, 5, 1, 10, 0, 7, 11, 15, 12, 6, 8, 13], [14, 10, 4, 12, 5, 6, 13, 2, 0, 1, 8, 7, 11, 3, 9, 15]]


In [65]:
test_problems = randomWalk_5step + randomWalk_10step + randomWalk_20step + randomWalk_40step + randomWalk_80step

### Breadth-First Search

In [68]:
TileSliding = Problem()
TileSliding.INITIAL = InitialState
TileSliding.IsGoal = lambda s: s==(1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0)
TileSliding.Actions = Actions
TileSliding.Result=Result
TileSliding.ActionCost = lambda s, a, sPrime: 1

Solutions = []
for s in test_problems:
  TileSliding.INITIAL = s
  ret, cost = BreadthFirstSearch(TileSliding)
  sol = Solution(ret)
  print (sol, ":", TileSliding.INITIAL)
  #print ("-----------------------")
  #print (TileSliding.INITIAL,':' )
  #print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost, '\n\n')
  #print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
#print ("-------")
#print (Solutions)

['d', 'r', 'r', 'd', 'd'] : [1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Length of solution:  5
Nodes Expanded= 40 


['d', 'r', 'r', 'd', 'r'] : [1, 2, 3, 4, 0, 6, 7, 8, 5, 9, 10, 12, 13, 14, 11, 15]
Length of solution:  5
Nodes Expanded= 37 


['d', 'd', 'r', 'r', 'r'] : [1, 2, 3, 4, 0, 6, 7, 8, 5, 10, 11, 12, 9, 13, 14, 15]
Length of solution:  5
Nodes Expanded= 30 


['l', 'l', 'u', 'u', 'r', 'r', 'r', 'd', 'd', 'd'] : [5, 1, 2, 3, 9, 6, 7, 4, 10, 11, 0, 8, 13, 14, 15, 12]
Length of solution:  10
Nodes Expanded= 2626 


['d', 'l', 'l', 'u', 'r', 'd', 'r', 'd', 'd', 'r'] : [5, 1, 0, 4, 2, 6, 3, 8, 9, 10, 7, 12, 13, 14, 11, 15]
Length of solution:  10
Nodes Expanded= 1555 


['l', 'd', 'l', 'u', 'u', 'r', 'r', 'r', 'd', 'd'] : [1, 2, 3, 4, 9, 5, 6, 7, 13, 11, 0, 8, 14, 10, 15, 12]
Length of solution:  10
Nodes Expanded= 2494 


['d', 'd', 'r', 'u', 'r', 'u', 'l', 'l', 'd', 'r', 'd', 'd', 'l', 'u', 'r', 'r', 'r', 'd'] : [0, 3, 7, 4, 2, 6, 5, 8, 1, 13, 10, 11, 14, 9, 15, 12]


### A* w/ Out of place

In [69]:
AStarFb = lambda n: n.PathCost + OutOfPlace(n.State, GoalState)

Solutions = []
for s in test_problems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarFb)
  sol = Solution(ret)
  print (sol, ":", TileSliding.INITIAL)
  #print ("-----------------------")
  #print (TileSliding.INITIAL,':' )
  #print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost, '\n\n')
  #print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
#print ("-------")
#print (Solutions)

['d', 'r', 'r', 'd', 'd'] : [1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Length of solution:  5
Nodes Expanded= 5 


['d', 'r', 'r', 'd', 'r'] : [1, 2, 3, 4, 0, 6, 7, 8, 5, 9, 10, 12, 13, 14, 11, 15]
Length of solution:  5
Nodes Expanded= 5 


['d', 'd', 'r', 'r', 'r'] : [1, 2, 3, 4, 0, 6, 7, 8, 5, 10, 11, 12, 9, 13, 14, 15]
Length of solution:  5
Nodes Expanded= 5 


['l', 'l', 'u', 'u', 'r', 'r', 'r', 'd', 'd', 'd'] : [5, 1, 2, 3, 9, 6, 7, 4, 10, 11, 0, 8, 13, 14, 15, 12]
Length of solution:  10
Nodes Expanded= 10 


['d', 'l', 'l', 'u', 'r', 'd', 'r', 'd', 'd', 'r'] : [5, 1, 0, 4, 2, 6, 3, 8, 9, 10, 7, 12, 13, 14, 11, 15]
Length of solution:  10
Nodes Expanded= 25 


['l', 'd', 'l', 'u', 'u', 'r', 'r', 'r', 'd', 'd'] : [1, 2, 3, 4, 9, 5, 6, 7, 13, 11, 0, 8, 14, 10, 15, 12]
Length of solution:  10
Nodes Expanded= 10 


['d', 'd', 'r', 'u', 'r', 'u', 'l', 'l', 'd', 'r', 'd', 'd', 'l', 'u', 'r', 'r', 'r', 'd'] : [0, 3, 7, 4, 2, 6, 5, 8, 1, 13, 10, 11, 14, 9, 15, 12]
Length of

### A* w/ Manhattan

In [70]:
AStarF = lambda n: n.PathCost + ManhattanDistance(n.State, GoalState)

Solutions = []
for s in test_problems:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarF)
  sol = Solution(ret)
  print (sol, ":", TileSliding.INITIAL)
  #print ("-----------------------")
  #print (TileSliding.INITIAL,':' )
  #print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost, '\n\n')
  #print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
#print ("-------")
#print (Solutions)

['d', 'r', 'r', 'd', 'd'] : [1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12]
Length of solution:  5
Nodes Expanded= 5 


['d', 'r', 'r', 'd', 'r'] : [1, 2, 3, 4, 0, 6, 7, 8, 5, 9, 10, 12, 13, 14, 11, 15]
Length of solution:  5
Nodes Expanded= 5 


['d', 'd', 'r', 'r', 'r'] : [1, 2, 3, 4, 0, 6, 7, 8, 5, 10, 11, 12, 9, 13, 14, 15]
Length of solution:  5
Nodes Expanded= 5 


['l', 'l', 'u', 'u', 'r', 'r', 'r', 'd', 'd', 'd'] : [5, 1, 2, 3, 9, 6, 7, 4, 10, 11, 0, 8, 13, 14, 15, 12]
Length of solution:  10
Nodes Expanded= 10 


['d', 'l', 'l', 'u', 'r', 'd', 'r', 'd', 'd', 'r'] : [5, 1, 0, 4, 2, 6, 3, 8, 9, 10, 7, 12, 13, 14, 11, 15]
Length of solution:  10
Nodes Expanded= 20 


['l', 'd', 'l', 'u', 'u', 'r', 'r', 'r', 'd', 'd'] : [1, 2, 3, 4, 9, 5, 6, 7, 13, 11, 0, 8, 14, 10, 15, 12]
Length of solution:  10
Nodes Expanded= 10 


['d', 'd', 'r', 'd', 'l', 'u', 'r', 'u', 'r', 'u', 'l', 'l', 'd', 'r', 'd', 'r', 'r', 'd'] : [0, 3, 7, 4, 2, 6, 5, 8, 1, 13, 10, 11, 14, 9, 15, 12]
Length of

# Results

The experiment compared how well BFS and AStar performed on different sizes and difficulties of sliding tile puzzles. For the simple 3x3 puzzle, both algorithms were able to solve even hard configurations, though AStar was faster and explored fewer states. But for the larger 4x4 puzzle, BFS struggled significantly. It took a very long time or failed to finish, especially on harder problems. In contrast, AStar still managed to solve most of the 4x4 puzzles, though it required more time and memory compared to the 3x3.

These results show that as the size and complexity of a problem increase, uninformed search methods like BFS become less practical due to their high resource usage. Heuristic-based approaches like AStar are more scalable because they can prioritize more promising paths. This highlights the importance of informed search in AI, especially when dealing with real-world problems that tend to be large and complex.